# Week 16 — Learning to Shape a Supersonic Pressure Signature
**FlowMLLab · Supersonic shape optimization · verified CFD + reduced-order machine learning**

> **Notebook philosophy.** This is a computational lesson, not a collection of audit scripts. Each section introduces the physics, derives the quantity used in the computation, runs a small reproducible calculation, and then asks what the result means.

## Learning objectives
By the end of this notebook you should be able to:

1. explain how supersonic motion produces a Mach cone and why multiple pressure waves can merge into a sonic-boom signature;
2. distinguish a **near-field pressure signature** from a **ground-level loudness metric**;
3. derive the fixed-volume body parameterization used in this module;
4. interpret the pressure coefficient \(C_p\) and convert it to \(\Delta p/p_\infty\);
5. reduce CFD waveforms with POD/SVD;
6. train a surrogate only on the training split and select it only with validation data;
7. quantify waveform, peak-pressure and pressure-drag errors separately;
8. use a surrogate for constrained design exploration;
9. verify the retained optimized design with independent CFD and mesh/off-design checks.

### Scope of the computed result
The accepted Week 16 result is an **axisymmetric, zero-incidence, inviscid teaching problem at \(M_\infty=1.8\)**. We optimize a pressure signature measured off the body at \(r/L=0.5\).

We do **not** propagate the waveform through the atmosphere and do **not** compute PLdB. Therefore a reduced near-field peak is evidence of pressure-signature shaping, not by itself proof of a quieter ground sonic boom.

In [ ]:
from pathlib import Path
import json, sys, subprocess, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "results/week16_lowboom").exists()),
    None
)

if ROOT is None and importlib.util.find_spec("google.colab") is not None:
    target = Path("/content/FlowMLLab_week16")
    if not target.exists():
        subprocess.run(
            ["git","clone","--depth","1","--filter=blob:none","--sparse",
             "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(target)],
            check=True
        )
        subprocess.run(
            ["git","-C",str(target),"sparse-checkout","set",
             "results/week16_lowboom","notebooks/week16","qa/week16"],
            check=True
        )
    ROOT = target

if ROOT is None:
    raise RuntimeError("Open this notebook from a clone of FlowMLLab.")

E = ROOT / "results/week16_lowboom"
R = E / "reference"

data = np.load(R / "clean_dataset_v801.npz", allow_pickle=False)
campaign = json.loads((R / "clean_campaign_audit.json").read_text())
design = json.loads((R / "weakwall_design_audit.json").read_text())
model_audit = json.loads((R / "clean_model_audit_v801.json").read_text())

assert campaign["passed"] and campaign["cases"] == 44
assert design["passed"]
assert model_audit["passed"]

print("Repository:", ROOT)
print("Verified clean CFD cases:", campaign["cases"])
print("Solver:", campaign["solver_version"])
print("Waveform points per case:", data["waveforms"].shape[1])

## 1. From supersonic motion to a pressure signature

The Mach number is

\[
M=\frac{U}{a},
\]

where \(U\) is the flight speed and \(a\) is the local speed of sound. When \(M>1\), information cannot propagate upstream. Small disturbances are confined by the **Mach cone**, whose half-angle is

\[
\mu=\sin^{-1}\!\left(\frac{1}{M}\right).
\]

For a real supersonic vehicle, compression and expansion waves are generated by changes in area, slope, lift, and volume distribution. During propagation these waves interact and can steepen or merge. Far from a conventional aircraft the pressure history often approaches an N-like waveform.

### Warm-up
For this Week 16 case, \(M_\infty=1.8\). Compute the Mach angle and compare it with Mach 1.2 and Mach 3.

In [ ]:
mach_values = np.array([1.2, 1.8, 3.0])
mach_angle_deg = np.degrees(np.arcsin(1.0 / mach_values))

display(pd.DataFrame({
    "Mach number": mach_values,
    "Mach angle [deg]": mach_angle_deg
}))

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(mach_values, mach_angle_deg, "o-")
ax.set_xlabel("Mach number, M")
ax.set_ylabel("Mach angle, mu [deg]")
ax.set_title("Mach cone narrows as Mach number increases")
ax.grid(alpha=0.25)
plt.show()

## 2. What quantity do we actually optimize?

The CFD solution gives pressure \(p(x,r)\). We nondimensionalize pressure with

\[
C_p
=
\frac{p-p_\infty}{q_\infty},
\qquad
q_\infty=\frac{1}{2}\rho_\infty U_\infty^2.
\]

For a calorically perfect gas,

\[
\rho_\infty U_\infty^2
=
\gamma p_\infty M_\infty^2,
\]

so

\[
\boxed{
\frac{\Delta p}{p_\infty}
=
\frac{\gamma M_\infty^2}{2}C_p
}
\]

with \(\Delta p=p-p_\infty\).

For \(M_\infty=1.8\) and \(\gamma=1.4\),

\[
\frac{\Delta p}{p_\infty}=2.268\,C_p.
\]

The Week 16 design objective is the **maximum \(C_p\)** along the off-body line \(r/L=0.5\), subject to a pressure-drag constraint.

In [ ]:
gamma = 1.4
M = 1.8
factor = 0.5 * gamma * M**2

cp_demo = data["waveforms"][0]
dp_over_p = factor * cp_demo

print(f"At M={M}, delta-p/p_inf = {factor:.3f} Cp")
print("Example peak Cp:", float(cp_demo.max()))
print("Example peak delta-p/p_inf:", float(dp_over_p.max()))

fig, ax = plt.subplots(figsize=(9,3.5))
ax.plot(data["x"], cp_demo, label="Cp")
ax.plot(data["x"], dp_over_p, label="delta-p / p_inf")
ax.set_xlabel("x/L")
ax.set_ylabel("Pressure measure")
ax.set_title("Same near-field waveform in two pressure normalizations")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 3. Fixed-volume body parameterization

We want to alter the pressure field without allowing the optimizer to win simply by shrinking the body.

Let \(s=x/L\), with \(0\le s\le1\). The unnormalized radius shape is

\[
f(s;a,b)
=
\sin(\pi s)
\exp\!\left[a(2s-1)+b\cos(2\pi s)\right].
\]

The physical radius is

\[
r(s)=c\,f(s;a,b).
\]

For a body of revolution,

\[
V
=
\pi L\int_0^1 r(s)^2\,ds.
\]

Therefore

\[
\boxed{
c
=
\sqrt{
\frac{V}
{\pi L\int_0^1 f(s;a,b)^2\,ds}
}
}.
\]

The two design variables \(a\) and \(b\) redistribute volume longitudinally; \(c\) restores the same total volume.

In [ ]:
L = 1.0
V_TARGET = np.pi * 0.06**2 / 2

def radius_profile(s, a, b, volume=V_TARGET, length=L):
    s = np.asarray(s)
    f = np.sin(np.pi*s) * np.exp(a*(2*s-1) + b*np.cos(2*np.pi*s))
    integral = np.trapezoid(f*f, s)
    c = np.sqrt(volume / (np.pi * length * integral))
    return c*f

s = np.linspace(0,1,2001)
examples = [(0.0,0.0), (0.30,0.15), (-0.30,-0.15)]
rows = []

fig, ax = plt.subplots(figsize=(10,3.2))
for a,b in examples:
    r = radius_profile(s,a,b)
    V = np.pi * L * np.trapezoid(r*r,s)
    rows.append({
        "a":a, "b":b, "volume":V,
        "relative volume error":abs(V/V_TARGET-1)
    })
    line, = ax.plot(s,r,label=f"a={a:+.2f}, b={b:+.2f}")
    ax.plot(s,-r,color=line.get_color())

ax.set_aspect("equal")
ax.set_xlabel("x/L")
ax.set_ylabel("r/L")
ax.set_title("Different shapes, the same volume")
ax.legend(ncol=3)
plt.show()

display(pd.DataFrame(rows))
assert max(row["relative volume error"] for row in rows) < 1e-5

## 4. The CFD model and its limits

The solver treats the body as **axisymmetric**, not as a planar 2-D airfoil. In conservative form the axisymmetric Euler equations may be written schematically as

\[
\frac{\partial (r\mathbf U)}{\partial t}
+
\frac{\partial (r\mathbf F_x)}{\partial x}
+
\frac{\partial (r\mathbf F_r)}{\partial r}
=
\mathbf S_r,
\]

where the radial source term accounts for cylindrical geometry.

For an ideal gas,

\[
p=(\gamma-1)
\left[
\rho E
-
\frac{\rho(u^2+v^2)}{2}
\right].
\]

### What is omitted?
- viscosity and skin friction;
- lift and angle of attack;
- wings, nacelles and propulsion;
- atmospheric stratification and absorption;
- nonlinear propagation to the ground;
- psychoacoustic loudness metrics.

The notebook isolates the coupling

\[
\text{shape}
\rightarrow
\text{near-field pressure waveform}
\rightarrow
\text{learned surrogate}
\rightarrow
\text{constrained design}.
\]

## 5. Verified CFD dataset

The clean student dataset contains 44 geometries recomputed with official SU2 8.0.1. The frozen split is

- 24 training cases,
- 6 validation cases,
- 8 test cases,
- 6 extrapolation cases.

All splits use the same two-parameter geometry family. Extrapolation therefore means parameter extrapolation, **not** generalization to a new aircraft family.

A model must never use test or extrapolation cases for fitting or model selection.

In [ ]:
split_counts = pd.Series(data["splits"]).value_counts()
display(split_counts.rename("number of cases"))

markers = {"train":"o", "validation":"s", "test":"^", "extrapolation":"x"}

fig, ax = plt.subplots(figsize=(7,5))
for split, marker in markers.items():
    m = data["splits"] == split
    ax.scatter(
        data["parameters"][m,0],
        data["parameters"][m,1],
        marker=marker, s=55, label=split
    )

ax.set_xlabel("shape parameter a")
ax.set_ylabel("shape parameter b")
ax.set_title("Frozen train / validation / test / extrapolation split")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

assert split_counts.to_dict() == {
    "train":24, "test":8, "validation":6, "extrapolation":6
}

## 6. Inspect the pressure waveforms before using machine learning

For every case we have

\[
\mathbf y_i =
\left[
C_p(x_1),\,C_p(x_2),\,\ldots,\,C_p(x_m)
\right].
\]

Before fitting a model, ask:

1. Where are the strongest compression peaks?
2. Do shocks shift in \(x\) as geometry changes?
3. Are amplitude and shock-location errors both important?
4. Does extrapolation visibly differ from the training family?

The waveform itself is the physical object of interest, so line plots are used rather than bar charts.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4.2))
for split in ["train","validation","test","extrapolation"]:
    ids = np.where(data["splits"] == split)[0]
    i = ids[len(ids)//2]
    ax.plot(
        data["x"], data["waveforms"][i],
        label=f"{split}: {data['names'][i]}"
    )

ax.axhline(0, lw=0.8)
ax.set_xlabel("x/L")
ax.set_ylabel("Cp")
ax.set_title("Representative near-field CFD pressure signatures")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
plt.show()

display(pd.DataFrame({
    "case": data["names"][:10],
    "split": data["splits"][:10],
    "peak Cp": data["waveforms"][:10].max(axis=1),
    "pressure drag": data["cd"][:10]
}))

## 7. Proper Orthogonal Decomposition (POD)

A pressure signature contains hundreds of spatial samples, but the waveform family can often be represented in a much smaller basis.

For the training matrix \(\mathbf Y\), subtract the training mean:

\[
\mathbf Y'=\mathbf Y-\overline{\mathbf Y}.
\]

Compute

\[
\mathbf Y'
=
\mathbf U\,\mathbf\Sigma\,\mathbf V^T.
\]

The rows of \(\mathbf V^T\) are spatial POD modes. A waveform is approximated as

\[
C_p(x;\boldsymbol\theta)
\approx
\overline C_p(x)
+
\sum_{k=1}^{K}
z_k(\boldsymbol\theta)\phi_k(x).
\]

The cumulative retained energy is

\[
E_K
=
\frac{\sum_{k=1}^{K}\sigma_k^2}
{\sum_j \sigma_j^2}.
\]

Only **training waveforms** are allowed to construct the basis.

In [ ]:
train = data["splits"] == "train"
Y_train = data["waveforms"][train]

mean_wave = Y_train.mean(axis=0)
Yc = Y_train - mean_wave

U, S, VT = np.linalg.svd(Yc, full_matrices=False)
energy = np.cumsum(S**2) / np.sum(S**2)

K = min(12, len(S))
Phi = VT[:K]

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(np.arange(1,len(S)+1), energy, "o-")
ax.axvline(K, ls="--", lw=1)
ax.set_xlabel("number of POD modes")
ax.set_ylabel("cumulative retained energy")
ax.set_ylim(0,1.01)
ax.set_title("Compression of the CFD waveform space")
ax.grid(alpha=0.2)
plt.show()

print(f"Retained energy with K={K}: {energy[K-1]:.8f}")

## 8. Surrogate learning: geometry to POD coefficients and drag

Instead of predicting every waveform sample directly, we learn

\[
(a,b)
\longrightarrow
\left[
z_1,\ldots,z_K,\log C_{D,p}
\right].
\]

We compare two compact models:

- **Ridge regression**, a regularized linear baseline;
- **MLP**, two 32-neuron tanh hidden layers.

The correct workflow is

\[
\text{training fit}
\rightarrow
\text{validation selection}
\rightarrow
\text{test evaluation}
\rightarrow
\text{extrapolation stress test}.
\]

The test set does not select the model.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor

X = data["parameters"]
Y = data["waveforms"]
CD = data["cd"]

def reduced_targets(mask):
    Z = (Y[mask] - mean_wave) @ Phi.T
    return np.c_[Z, np.log(CD[mask])]

x_scaler = StandardScaler().fit(X[train])
y_scaler = StandardScaler().fit(reduced_targets(train))

Xtr = x_scaler.transform(X[train])
Ttr = y_scaler.transform(reduced_targets(train))

models = {
    "ridge": Ridge(alpha=1e-2),
    "mlp": MLPRegressor(
        hidden_layer_sizes=(32,32),
        activation="tanh",
        solver="lbfgs",
        alpha=1e-2,
        max_iter=3000,
        random_state=16,
        tol=1e-7
    )
}

for model in models.values():
    model.fit(Xtr,Ttr)

def predict(model, Xq):
    Tout = y_scaler.inverse_transform(
        model.predict(x_scaler.transform(np.atleast_2d(Xq)))
    )
    Z = Tout[:,:K]
    wave = mean_wave + Z @ Phi
    cd = np.exp(Tout[:,K])
    return wave, cd

print("Models fitted only on", int(train.sum()), "training cases.")

## 9. Error metrics answer different questions

### Whole-waveform relative error

\[
\epsilon_{\mathrm{wave}}
=
\frac{\lVert \widehat{\mathbf y}-\mathbf y\rVert_2}
{\lVert \mathbf y\rVert_2}.
\]

### Peak-pressure relative error

\[
\epsilon_{\mathrm{peak}}
=
\left|
\frac{\max\widehat C_p-\max C_p}
{\max C_p}
\right|.
\]

### Pressure-drag relative error

\[
\epsilon_D
=
\left|
\frac{\widehat C_{D,p}-C_{D,p}}
{C_{D,p}}
\right|.
\]

A shifted shock can have a modest peak error but a large waveform error. A small aggregate error can also hide one poor individual prediction.

In [ ]:
def metrics(model, split):
    m = data["splits"] == split
    wp, cdp = predict(model, X[m])
    wt = Y[m]
    cdt = CD[m]

    per_wave = np.linalg.norm(wp-wt,axis=1) / np.linalg.norm(wt,axis=1)
    per_peak = np.abs(wp.max(axis=1)/wt.max(axis=1)-1)
    per_drag = np.abs(cdp/cdt-1)

    return {
        "cases": int(m.sum()),
        "wave L2": float(np.linalg.norm(wp-wt)/np.linalg.norm(wt)),
        "mean peak error": float(per_peak.mean()),
        "mean drag error": float(per_drag.mean()),
        "worst waveform error": float(per_wave.max())
    }

rows = []
for name,model in models.items():
    for split in ["validation","test","extrapolation"]:
        rows.append({"model":name,"split":split,**metrics(model,split)})

scores = pd.DataFrame(rows)
display(scores)

val = scores[scores["split"]=="validation"].copy()
val["selection score"] = val["mean peak error"] + val["mean drag error"]
selected_name = val.loc[val["selection score"].idxmin(),"model"]
selected = models[selected_name]

print("Selected using validation only:", selected_name)

In [ ]:
test = data["splits"] == "test"
pred_test, pred_cd_test = predict(selected, X[test])
truth_test = Y[test]
names_test = data["names"][test]

fig, axes = plt.subplots(4,2,figsize=(11,12),sharex=True)
per_case = []

for ax, name, truth, pred, cd_t, cd_p in zip(
    axes.flat, names_test, truth_test, pred_test, CD[test], pred_cd_test
):
    ewave = np.linalg.norm(pred-truth)/np.linalg.norm(truth)
    epeak = abs(pred.max()/truth.max()-1)
    edrag = abs(cd_p/cd_t-1)
    per_case.append([name,ewave,epeak,edrag])

    ax.plot(data["x"], truth, lw=2, label="SU2 CFD")
    ax.plot(data["x"], pred, "--", label=f"{selected_name} surrogate")
    ax.set_title(f"{name} | wave error={100*ewave:.1f}%")
    ax.set_ylabel("Cp")
    ax.grid(alpha=0.2)

for ax in axes[-1,:]:
    ax.set_xlabel("x/L")
axes.flat[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

case_errors = pd.DataFrame(
    per_case,
    columns=["case","wave L2","peak error","drag error"]
)
display(case_errors.sort_values("wave L2",ascending=False))

## 10. Retained audited neural model

The notebook fit above is pedagogical. The repository also contains a separately retained clean-label model with a frozen identity.

Its audit records:

- dataset and checkpoint hashes;
- train-only scaling and POD basis;
- architecture and seed;
- same-mesh split errors;
- finer-mesh CFD comparison;
- worst individual waveform error.

The finer-mesh comparison is retrospective because those CFD solutions already existed before this fit. It is therefore useful evidence, but not a new prospective blind experiment.

In [ ]:
keys = [
    "wave_relative_l2",
    "peak_mean_relative_error",
    "drag_mean_relative_error",
    "worst_case_wave_relative_l2"
]

same_mesh = pd.DataFrame({
    split: {k:100*v[k] for k in keys}
    for split,v in model_audit["same_mesh"].items()
}).T

same_mesh.columns = [
    "wave L2 [%]",
    "mean peak error [%]",
    "mean drag error [%]",
    "worst waveform error [%]"
]
display(same_mesh)

display(pd.Series(
    {k:100*model_audit["finer_mesh"][k] for k in keys},
    name="retained model vs finer CFD [%]"
))

print("Checkpoint SHA-256:", model_audit["checkpoint_sha256"])
print("Audit passed:", model_audit["passed"])

## 11. Constrained design exploration

The design problem is

\[
\min_{a,b}
\quad
\max_x C_p(x;a,b)
\]

subject to

\[
C_{D,p}(a,b)
\le
1.02\,C_{D,p}^{\mathrm{baseline}},
\]

while fixed volume is enforced by the geometry parameterization.

A surrogate may evaluate thousands of candidates cheaply, but an optimizer also searches for weaknesses in the surrogate. Therefore the result of this section is only a **proposal** until it is recomputed with CFD.

In [ ]:
baseline_run = design["runs"][0]
baseline_cd = baseline_run["cd_pressure"]
baseline_peak = baseline_run["peak_cp"]
drag_limit = 1.02 * baseline_cd

a_min, b_min = X[train].min(axis=0)
a_max, b_max = X[train].max(axis=0)

aa = np.linspace(a_min,a_max,80)
bb = np.linspace(b_min,b_max,80)
grid = np.array([(a,b) for a in aa for b in bb])

wave_g, cd_g = predict(selected,grid)
peak_g = wave_g.max(axis=1)
feasible = cd_g <= drag_limit

best_i = np.where(feasible)[0][np.argmin(peak_g[feasible])]

display(pd.Series({
    "a": grid[best_i,0],
    "b": grid[best_i,1],
    "predicted peak Cp": peak_g[best_i],
    "predicted pressure drag": cd_g[best_i],
    "predicted peak reduction [%]":
        100*(1-peak_g[best_i]/baseline_peak)
},name="Notebook surrogate proposal"))

fig, ax = plt.subplots(figsize=(7,5))
sc = ax.scatter(grid[:,0],grid[:,1],c=peak_g,s=12)
ax.scatter(
    grid[best_i,0],grid[best_i,1],
    marker="*",s=180,label="best feasible surrogate point"
)
ax.set_xlabel("a")
ax.set_ylabel("b")
ax.set_title("Surrogate-predicted peak pressure over design space")
ax.legend()
plt.colorbar(sc,ax=ax,label="predicted peak Cp")
plt.show()

## 12. Fresh CFD verification of the retained optimized design

A retained candidate was recomputed with SU2 8.0.1 instead of being accepted from the surrogate.

At the two design meshes, the measured peak reductions are approximately

\[
21.19\%
\quad\text{and}\quad
20.66\%.
\]

The pressure-drag changes are approximately

\[
-3.80\%
\quad\text{and}\quad
-3.52\%.
\]

Thus, within this teaching problem, the retained candidate reduces both the near-field pressure peak and pressure drag.

Mesh sensitivity must still be shown separately: similar scalar peaks do not prove identical waveforms.

In [ ]:
pairs = pd.DataFrame([
    {
        "mesh level": p["level"],
        "peak reduction [%]": 100*p["peak_reduction"],
        "drag change [%]": 100*p["drag_change"],
        "drag ratio": p["drag_ratio"]
    }
    for p in design["design_pairs"]
])
display(pairs)

mesh = pd.DataFrame([
    {
        "design": m["design"],
        "peak mesh change [%]": 100*m["peak_relative_change"],
        "drag mesh change [%]": 100*m["drag_relative_change"],
        "waveform mesh L2 change [%]": 100*m["wave_relative_l2_change"]
    }
    for m in design["mesh_sensitivity"]
])
display(mesh)

assert design["checks"]["peak_reduction_above_5pct_at_both_design_meshes"]
assert design["checks"]["drag_ratio_at_most_1p02_at_both_design_meshes"]

print("All declared design checks passed:", design["passed"])

In [ ]:
with np.load(R / "weakwall_design_test.npz",allow_pickle=False) as ev:
    waves = ev["waveforms"]
    xev = ev["x"]

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(xev,waves[0],lw=2,label="baseline, mesh 1.5")
ax.plot(xev,waves[1],lw=2,label="optimized, mesh 1.5")
ax.plot(xev,waves[2],"--",label="baseline, mesh 2")
ax.plot(xev,waves[3],"--",label="optimized, mesh 2")
ax.set_xlabel("x/L")
ax.set_ylabel("Cp")
ax.set_title("Fresh CFD verification: pressure-signature reshaping")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 13. Off-design robustness

A design optimized at one Mach number can fail away from that condition. The retained geometry was therefore recomputed at \(M=1.7\) and \(M=1.9\).

These are **CFD calculations of fixed geometries**, not surrogate predictions.

This tests local operating-condition robustness; it does not represent atmospheric uncertainty or a full flight envelope.

In [ ]:
off = pd.DataFrame([
    {
        "Mach": p["mach"],
        "peak reduction [%]": 100*p["peak_reduction"],
        "drag change [%]": 100*p["drag_change"]
    }
    for p in design["offdesign_pairs"]
])
display(off)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(off["Mach"],off["peak reduction [%]"],"o-",label="peak reduction")
ax.plot(off["Mach"],-off["drag change [%]"],"s-",label="magnitude of drag reduction")
ax.set_xlabel("Mach number")
ax.set_ylabel("improvement [%]")
ax.set_title("Retained design at nearby Mach numbers")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 14. Why this is not yet a ground sonic-boom calculation

A near-field pressure signature is only the initial condition for atmospheric propagation.

A common weak-shock framework is a generalized Burgers-type equation. Schematically,

\[
\frac{\partial p'}{\partial s}
+
\beta p'\frac{\partial p'}{\partial\tau}
=
\delta\frac{\partial^2p'}{\partial\tau^2}
+
\mathcal G[p']
+
\mathcal A[p'],
\]

where

- \(s\) is propagation distance;
- \(\tau\) is retarded time;
- \(p'\) is acoustic overpressure;
- the nonlinear term steepens the waveform;
- the diffusion-like term represents thermoviscous absorption;
- \(\mathcal G\) represents geometric spreading;
- \(\mathcal A\) may include stratification, molecular relaxation, wind, and related atmospheric effects.

Only after propagation can a ground waveform be converted into psychoacoustic metrics such as PLdB.

Therefore

\[
\boxed{
\text{lower near-field peak}
\neq
\text{automatically lower ground PLdB}
}
\]

The defensible Week 16 statement is:

> Within this axisymmetric inviscid teaching problem, the retained optimized geometry produces a smaller verified near-field pressure peak while satisfying the pressure-drag constraint.

## 15. Verification and validation: three different questions

### Numerical verification
**Did we solve the chosen equations correctly?**

Examples include residual reduction, iterative convergence, mesh sensitivity, physical consistency checks, and the independent Taylor–Maccoll cone solution.

### Physical validation
**Do the equations and numerical model reproduce measured reality adequately for the intended quantity?**

That requires appropriate experimental evidence and uncertainty treatment.

### Machine-learning validation
**Does the surrogate predict withheld CFD solutions?**

That requires frozen data splits, train-only preprocessing, individual as well as aggregate errors, extrapolation tests, and fresh CFD evaluation of optimized candidates.

A neural network can predict an inaccurate CFD dataset extremely well. Success at one layer never automatically validates the next layer.

In [ ]:
display(pd.DataFrame([
    ["Clean CFD cases", campaign["cases"],
     "all declared numerical/full-field checks passed"],
    ["Train / val / test / extrap", "24 / 6 / 8 / 6",
     "same two-parameter family"],
    ["Finer-mesh design peak reduction",
     f"{100*design['design_pairs'][1]['peak_reduction']:.2f}%",
     "direct CFD"],
    ["Finer-mesh design drag change",
     f"{100*design['design_pairs'][1]['drag_change']:.2f}%",
     "direct CFD"],
    ["Ground PLdB", "not computed",
     "outside accepted Week 16 scope"]
],columns=["item","result","interpretation"]))

## 16. Exercises

### Exercise 1 — Mach angle
Compute the Mach angle for \(M=1.4,\;1.6,\;2.0,\;2.5\). Explain why the cone narrows.

### Exercise 2 — Pressure normalization
At \(M=1.8\), convert \(C_p=0.03\) to \(\Delta p/p_\infty\). Repeat at \(M=1.6\).

### Exercise 3 — Geometry
Choose two new \((a,b)\) pairs and verify that fixed volume is preserved to within \(10^{-5}\) relative error.

### Exercise 4 — POD truncation
Repeat the surrogate workflow with \(K=4,\;8,\;12\). Compare retained energy and test waveform error.

### Exercise 5 — Data leakage
Build the POD basis using all 44 cases and compare the apparent test error with the correct train-only pipeline. Explain why the former result is invalid.

### Exercise 6 — Worst-case prediction
Find the test geometry with the largest waveform error. Decide whether amplitude error, shock-location error, or both dominate.

### Exercise 7 — Optimization
Tighten the drag ratio from 1.02 to 1.00. Re-run the surrogate scan and compare the selected geometry.

### Exercise 8 — Claim discipline
Write one sentence justified by this notebook and one sentence that would be an overclaim. Explicitly distinguish near-field pressure from ground loudness.

---

## Take-away

The central scientific-ML loop is

\[
\boxed{
\text{physics}
\rightarrow
\text{verified CFD}
\rightarrow
\text{POD}
\rightarrow
\text{surrogate}
\rightarrow
\text{constrained proposal}
\rightarrow
\text{fresh CFD verification}
}
\]

The surrogate accelerates exploration. Verification, validation, and careful scientific claims remain essential.

### Supporting Week 16 material
README.md, MODEL_VALIDATION_GUIDE.md, NASA_REFERENCE_GUIDE.md, REFERENCE_REPRODUCTION.md, TEACHING_REVIEW.md, the CFD reproduction guide, and the retained numerical-evidence report are all distributed with Week 16.

**Reference context:** Taylor–Maccoll cone verification; SU2 governing equations; NASA/AIAA sonic-boom benchmark material; Zheng et al. (2026), Aerospace Science and Technology, DOI 10.1016/j.ast.2026.113218.